# Tutorial 06 — Mechanical Homeostasis

This notebook computes all trajectories from the local source package. It does not load committed figures.

## 1. Environment and imports

In [ ]:
LANGUAGE = "en"

import importlib.util
import os
import sys
from pathlib import Path


def _is_repository_root(path: Path) -> bool:
    return (path / "pyproject.toml").is_file() and (
        path / "src" / "biomechanics_tutorials"
    ).is_dir()


def _find_repository_root() -> Path:
    candidates = []
    installed_spec = importlib.util.find_spec("biomechanics_tutorials")
    if installed_spec is not None and installed_spec.origin:
        package_file = Path(installed_spec.origin).resolve()
        if len(package_file.parents) >= 3:
            candidates.append(package_file.parents[2])
    environment_root = os.environ.get("BMRT_ROOT")
    if environment_root:
        candidates.append(Path(environment_root).expanduser())
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    home = Path.home()
    for directory in [home, home / "Desktop", home / "Documents", home / "Downloads"]:
        candidates.append(directory / "Biomechanics-Research-Tutorials")
        if directory.is_dir():
            candidates.extend(directory.glob("Biomechanics-Research-Tutorials*"))
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_repository_root(candidate):
            return candidate
    raise RuntimeError(
        "Repository root was not found. Extract the complete repository or install it with python -m pip install -e .[dev]"
    )


REPOSITORY_ROOT = _find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import numpy as np
import matplotlib.pyplot as plt
import biomechanics_tutorials
from biomechanics_tutorials.mechanical_homeostasis import (
    ConstituentTurnoverParameters,
    HomeostasisParameters,
    analytical_capacity_response,
    generate_load_protocol,
    homeostasis_metrics,
    simulate_constituent_turnover,
    simulate_scalar_homeostasis,
    simulate_vessel_homeostasis,
    vessel_equilibrium,
)

print("Repository root:", REPOSITORY_ROOT)
print("Kernel executable:", sys.executable)
print("Local package:", Path(biomechanics_tutorials.__file__).resolve())

## 2. Analytical verification

In [ ]:
time = np.linspace(0.0, 15.0, 3001)
params = HomeostasisParameters(adaptation_rate=0.35)
numerical = simulate_scalar_homeostasis(time, 1.55, 1.0, params)
analytical = analytical_capacity_response(time, 1.55, 1.0, 1.0, 0.35)
print("maximum capacity error:", np.max(np.abs(numerical["capacity"] - analytical)))
plt.plot(time, analytical, label="analytical")
plt.plot(time, numerical["capacity"], "--", label="numerical")
plt.xlabel("t")
plt.ylabel("c")
plt.legend();

## 3. Disturbance protocols

In [ ]:
time = np.linspace(0.0, 28.0, 2801)
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, kind in zip(axes.flat, ["step", "pulse", "ramp", "cyclic"], strict=True):
    load = generate_load_protocol(time, kind, amplitude=0.45, onset=4.0, duration=7.0, period=5.0)
    result = simulate_scalar_homeostasis(time, load)
    ax.plot(time, result["load"], label="L")
    ax.plot(time, result["stress"], label="sigma/sigma_h")
    ax.plot(time, result["capacity"], label="c")
    ax.set_title(kind)
    ax.legend(fontsize=8)
plt.tight_layout();

## 4. Delay and stability

In [ ]:
time = np.linspace(0.0, 35.0, 3501)
load = generate_load_protocol(time, "step", amplitude=0.5, onset=3.0)
for delay in [0.0, 1.0, 2.5, 4.0]:
    result = simulate_scalar_homeostasis(
        time,
        load,
        parameters=HomeostasisParameters(adaptation_rate=0.6, delay=delay, capacity_min=0.2, capacity_max=4.0),
    )
    plt.plot(time, result["stress"], label=f"delay={delay}")
plt.axhline(1.0, linestyle=":")
plt.xlabel("t")
plt.ylabel("sigma/sigma_h")
plt.legend();

## 5. Constituent turnover

In [ ]:
time = np.linspace(0.0, 35.0, 1401)
error = np.zeros_like(time)
error[(time >= 6.0) & (time < 16.0)] = 0.2
constituents = [
    ConstituentTurnoverParameters("collagen", 0.60, 24.0, 1.7, 0.1, 1.08),
    ConstituentTurnoverParameters("smooth muscle", 0.28, 10.0, 1.2, 0.2, 1.05),
    ConstituentTurnoverParameters("ground matrix", 0.12, 4.0, 0.7, 0.3, 1.00),
]
turnover = simulate_constituent_turnover(time, error, constituents)
for row, name in enumerate(turnover["names"]):
    plt.plot(time, turnover["mass"][row], label=name)
plt.xlabel("t")
plt.ylabel("mass")
plt.legend();

## 6. Vascular homeostasis

In [ ]:
time = np.linspace(0.0, 70.0, 3501)
pressure = np.where(time < 4.0, 1.0, 1.35)
flow = np.where(time < 4.0, 1.0, 1.60)
vessel = simulate_vessel_homeostasis(time, pressure, flow)
print("analytical equilibrium:", vessel_equilibrium(1.35, 1.60))
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(time, vessel["radius"], label="r/r_h")
axes[0].plot(time, vessel["thickness"], label="h/h_h")
axes[1].plot(time, vessel["shear_ratio"], label="tau/tau_h")
axes[1].plot(time, vessel["wall_stress_ratio"], label="sigma/sigma_h")
for ax in axes:
    ax.legend()
plt.tight_layout();

## 7. Recovery metrics

In [ ]:
time = np.linspace(0.0, 30.0, 3001)
load = generate_load_protocol(time, "step", amplitude=0.5, onset=3.0)
result = simulate_scalar_homeostasis(time, load)
active = time >= 3.0
homeostasis_metrics(time[active], result["true_error"][active])

### Interpretation
A recovered target is meaningful only together with the feedback law, sensing model, delays, limits, and verification protocol.